# Lesson 01 · Perceptron: one neuron that reads handwritten digits

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/01-perceptron-en.ipynb)

This notebook accompanies [Lesson 01 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/perceptron/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/01-perceptron.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
Using only numpy, with no machine-learning library, we write a neuron ourselves that learns to answer the question
**“is this image a 7 or a 1?”** on real MNIST data.

**What you'll do:**
1. Turn a 28×28 image into a vector $x$ of 784 inputs.
2. Compute $z = \sum_i x_i w_i + b$ (784 multiply-accumulate operations) and decide with a step function.
3. Train with Rosenblatt's learning rule: update the weights only when the prediction is wrong.
4. Read the learned weights as an image, and see where the model goes wrong.
5. See the limit of a single neuron (XOR), and step up from one neuron to a layer, $z = xW + b$.

**Background:** basic Python and numpy (arrays, `@`, indexing). **Run time:** under a minute on a CPU,
including the first MNIST download (~11 MB).

The notation follows the *Neural Computation* chapter of [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html).
Every number in the notebook is computed by the cells below, and matches the numbers in the reel and the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands (13,007; 99.35%). That is the same point Python uses
when you *type* a decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with distinct shapes: "1" = circle, "7" = square
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every linear axis (colorbars included) in the English format; call right before
    plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(tick))


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.993527))

## 2 · The MNIST data

MNIST (LeCun et al., 1998) has 70,000 grayscale 28×28 images of handwritten digits: 60,000 training images and
10,000 test images. The cell below downloads the 4 original files from a public mirror and checks their MD5 hashes,
so everyone who runs it gets exactly the same data.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
print("training:", train_img.shape, " test:", test_img.shape, " dtype:", train_img.dtype)

The figure below shows the first 20 training images, each with its true label above it.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for ax, img, lab in zip(axes.flat, train_img, train_lab):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
fmt_axes(plt.gcf())
plt.show()

## 3 · From image to vector $x$

A neuron doesn't “see” an image; it receives a sequence of numbers. We lay the image's 28 rows end to end into the
vector $x = [x_1, \dots, x_{784}]$, and divide by 255 so that every $x_i$ lies in $[0, 1]$: 0 is the white background,
1 the darkest stroke.

We keep only the images of 7s and 1s, and give the label $y = 1$ to “7” and $y = 0$ to “1”.

In [ ]:
POS, NEG = 7, 1  # y = 1 means "7", y = 0 means "1"


def binary_set(images, labels, pos=POS, neg=NEG):
    keep = (labels == pos) | (labels == neg)
    X = images[keep].reshape(-1, 784) / 255.0  # each row is one image: 784 numbers in [0, 1]
    y = (labels[keep] == pos).astype(int)
    return X, y, np.nonzero(keep)[0]  # the original indices in MNIST, to look the images up again


Xtr, ytr, idx_tr = binary_set(train_img, train_lab)
Xte, yte, idx_te = binary_set(test_img, test_lab)
print(
    f"training: {num(len(ytr), 0)} images ({num(ytr.sum(), 0)} 7s)  ·  test: {num(len(yte), 0)} images ({num(yte.sum(), 0)} 7s)"
)
print("one image is a vector of length", Xtr[0].shape[0])
assert (len(ytr), ytr.sum(), len(yte), yte.sum()) == (13007, 6265, 2163, 1028)

Let's look at one image in both forms: the 28×28 grid, and the strip of 784 numbers (row 1 of the image is the first
28 numbers, row 2 the next 28…). Most $x_i$ are 0 because they are background; only the pixels with ink are nonzero.

In [ ]:
x = Xtr[0]
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 2.6), gridspec_kw={"width_ratios": [1, 4]})
a.imshow(x.reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
a.set_title(f"28×28 image of a {POS if ytr[0] else NEG}")
a.axis("off")
b.plot(np.arange(1, 785), x, color=BLUE, lw=1)
b.set(xlabel="index i (1…784)", ylabel="xᵢ", xlim=(1, 784), title="the same image, laid out as the vector x")
fmt_axes(plt.gcf())
plt.show()
print(f"{(x > 0).sum()} of the 784 pixels have ink")

## 4 · One neuron: 784 multiply-accumulate operations

The perceptron gives each input $x_i$ a weight $w_i$, adds everything up together with the bias $b$, then decides:

$$z = \sum_{i=1}^{784} x_i\, w_i + b, \qquad \hat y = f(z) = \begin{cases} 1 \;(\text{“7”}) & z \ge 0 \\ 0 \;(\text{“1”}) & z < 0 \end{cases}$$

Each term $x_i w_i$ is one **multiply-accumulate (MAC) operation**: multiply, then add to the running total.
Writing it as a loop makes that explicit; numpy does the same work with one dot product, `x @ w`.

In [ ]:
rng = np.random.default_rng(42)
w_demo = rng.normal(0, 0.1, 784)  # random weights, only to illustrate the computation
b_demo = 0.5


def neuron_loop(x, w, b):
    z, macs = b, 0
    for xi, wi in zip(x, w):
        z += xi * wi  # one multiply-accumulate
        macs += 1
    return z, macs


def step(z):
    return (z >= 0).astype(int)


z_loop, macs = neuron_loop(x, w_demo, b_demo)
z_vec = x @ w_demo + b_demo
assert np.isclose(z_loop, z_vec)
print(f"z = {num(z_vec, 4)} after {macs} MACs  →  ŷ = {step(np.array(z_vec))}")

In [ ]:
t0 = time.perf_counter()
for x_ in Xte[:200]:
    neuron_loop(x_, w_demo, b_demo)
t_loop = (time.perf_counter() - t0) / 200
t0 = time.perf_counter()
Xte @ w_demo + b_demo  # the whole test set at once
t_vec = (time.perf_counter() - t0) / len(Xte)
print(
    f"Python loop: {num(t_loop * 1e6, 0)} µs/image  ·  numpy: {num(t_vec * 1e6, 2)} µs/image  (~{num(t_loop / t_vec, 0)} times faster)"
)

The computation is exactly the same; the difference is that numpy runs the 784 MACs in optimized machine code, and
processes all 2,163 images as one matrix multiplication. This is why neural networks are written as matrix
multiplications (section 10).

### 🤔 Predict first
Before learning, every weight is 0: $w = 0$, $b = 0$. What does the neuron answer for each image then,
and what percentage of the test set does it get right?

In [ ]:
w0, b0 = np.zeros(784), 0.0
pred0 = step(Xte @ w0 + b0)
print("distinct predictions:", np.unique(pred0), " →  accuracy", pct((pred0 == yte).mean()))

With $w = 0$, $z = 0$ for every image, and since $f(0) = 1$, the neuron answers “7” for all of them. The accuracy is
exactly the fraction of 7s in the test set: 1,028 / 2,163. This is the baseline to beat.

## 5 · Learning from mistakes: Rosenblatt's learning rule (1958)

Go through the training images one at a time, in random order. If the prediction is right, do nothing. If it is
wrong, update:

$$w_i \leftarrow w_i + \eta\,(y - \hat y)\,x_i, \qquad b \leftarrow b + \eta\,(y - \hat y)$$

- The image is a 7 ($y = 1$) but the prediction is “1” ($\hat y = 0$): $y - \hat y = +1$, so **add** $\eta x$ to $w$,
  and next time this image's $z$ is larger.
- The image is a 1 but the prediction is “7”: $y - \hat y = -1$, so **subtract** $\eta x$, and $z$ gets smaller.

$\eta$ (the learning rate) is the step size. We use $\eta = 0.1$ and one epoch (one pass) over the 13,007 images,
as the reel does.

In [ ]:
def train(X, y, eta=0.1, seed=0, epochs=1, X_eval=None, y_eval=None):
    """The perceptron learning rule. Returns w, b and a log of the updates (with the test accuracy after each one)."""
    w, b = np.zeros(X.shape[1]), 0.0
    order = np.random.default_rng(seed).permutation(len(y))
    log = {"seen": [], "i": [], "z": [], "acc": [], "epoch_errors": []}

    def evaluate():
        if X_eval is not None:
            log["acc"].append(float((step(X_eval @ w + b) == y_eval).mean()))

    evaluate()
    seen = 0
    for _ in range(epochs):
        errors = 0
        for i in order:
            seen += 1
            z = X[i] @ w + b
            y_hat = int(z >= 0)
            if y_hat != y[i]:
                d = y[i] - y_hat  # +1 or −1
                w += eta * d * X[i]
                b += eta * d
                errors += 1
                log["seen"].append(seen)
                log["i"].append(int(i))
                log["z"].append(float(z))
                evaluate()
        log["epoch_errors"].append(errors)
        if errors == 0:  # an epoch without a single mistake: the training set is perfectly separated
            break
    return w, b, log


t0 = time.perf_counter()
w, b, log = train(Xtr, ytr, eta=0.1, seed=0, X_eval=Xte, y_eval=yte)
pred = step(Xte @ w + b)
print(f"{len(log['seen'])} updates in {num(len(ytr), 0)} images ({num((time.perf_counter() - t0) * 1000, 0)} ms)")
print(
    f"test accuracy: {pct(log['acc'][0])} → {pct((pred == yte).mean())}"
    f"  ({num((pred == yte).sum(), 0)}/{num(len(yte), 0)} images)  ·  b = {num(b, 2)}"
)
assert len(log["seen"]) == 146 and (pred == yte).sum() == 2149, "differs from the reel and the demo: check the data"

Only **146 updates** (about 1% of the images) are enough to take the accuracy from 47.53% to **99.35%**, exactly the
numbers in the reel and the demo. Most of the updates come early: after a few hundred images, the neuron already
classifies almost everything correctly.

The plot below shows the test accuracy after each update (the horizontal axis counts updates, not images seen).
The dashed line is the 47.53% baseline of $w = 0$.

In [ ]:
acc = np.array(log["acc"])
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(np.arange(len(acc)), 100 * acc, color=BLUE, lw=2)
ax.axhline(100 * acc[0], color=GREY, lw=1, ls="--")
ax.set(xlabel="number of updates", ylabel="correct on the test set (%)", ylim=(40, 101), xlim=(0, len(acc) - 1))
ax.set_title("Accuracy after each update (η = 0.1, seed 0)")
ax.grid(axis="y", alpha=0.3)
fmt_axes(plt.gcf())
plt.show()
print("first 10 updates:", " · ".join(pct(a, 1) for a in acc[:11]))
seen = np.array(log["seen"])
print(
    f"which images caused updates: the first 10 within the first {seen[9]} images; {(seen <= 1000).sum()} within the first 1,000 images; "
    f"the last at image no. {num(seen[-1], 0)}"
)

The curve doesn't rise steadily: an update that fixes one image can break many others
(the 4th update drags the accuracy from 91.9% down to 63.4%). The rule looks at only one image at a time.

### The first 4 updates, up close
Each column is one update: the misclassified image (top row) and $w$ right after the update (bottom row; blue =
positive, orange = negative). Since $w$ starts at 0, $w$ after the first update is exactly $-\eta$ times that image.

In [ ]:
def show_w(ax, w, title=None, vmax=None):
    vmax = vmax or np.abs(w).max() or 1
    im = ax.imshow(w.reshape(28, 28), cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


w_step = np.zeros(784)
b_step = 0.0
fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
for k in range(4):
    i, z = log["i"][k], log["z"][k]
    d = ytr[i] - int(z >= 0)
    w_step += 0.1 * d * Xtr[i]
    b_step += 0.1 * d
    axes[0, k].imshow(Xtr[i].reshape(28, 28), cmap="gray_r")
    axes[0, k].set_title(
        f"image no. {log['seen'][k]}: a {POS if ytr[i] else NEG}\nz = {num(z, 2)} → predicts “{POS if z >= 0 else NEG}”",
        fontsize=10,
    )
    axes[0, k].axis("off")
    show_w(axes[1, k], w_step, f"w after update {k + 1} ({'+' if d > 0 else '−'}η·x)")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

The first update happens right at the first image: a 1 with $z = 0$ is predicted “7” (because $f(0) = 1$), so
$0.1$ times that 1's strokes is subtracted from $w$. The second image is another 1 and $z < 0$, so it is classified
correctly and there is no update.

## 6 · Reading the weights as an image

Each $w_i$ belongs to one pixel, so $w$ can be drawn as a 28×28 image.
**Blue** ($w_i > 0$): ink here pushes $z$ up, that is, toward “7”. **Orange** ($w_i < 0$): pulls $z$ down, toward “1”.

### 🤔 Predict first
Where do you think the blue and the orange regions will be? And why do some pixels have $w_i$ exactly 0?

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 3.6))
im = show_w(ax, w, "w after 146 updates")
fig.colorbar(im, ax=ax, fraction=0.046)
fmt_axes(plt.gcf())
plt.show()
touched = (Xtr[log["i"]] > 0).any(axis=0)
print(f"{(w > 0).sum()} positive weights, {(w < 0).sum()} negative, {(w == 0).sum()} exactly 0")
print(
    f"pixels never inked in the 146 misclassified images: {(~touched).sum()} — exactly the weights that are 0: {np.all((w == 0) == ~touched)}"
)

The orange region is a vertical band in the middle of the image, where the stroke of a 1 stands, plus a patch at the
bottom. The blue regions lie on both sides of that band, where a 7 has ink and a 1 doesn't: the horizontal bar and
the diagonal stroke. This picture is not an “average” 7: it is the sum of the 146 misclassified images, added or
subtracted, so it keeps only the strokes that help *tell the two classes apart*.

The zero weights are mostly around the border. Each update only adds $\pm\eta x$, so a pixel that has no ink in any
of the misclassified images keeps the weight of 0 it started with.

### How one image is decided
The individual products $x_i w_i$ show how much each pixel contributes to $z$. Take a “typical” 7 and a “typical” 1
from the test set (the image whose $z$ is closest to the median of its class; these are also the two images the reel
and the demo use):

In [ ]:
z_te = Xte @ w + b


def typical(cls):
    idx = np.nonzero(yte == cls)[0]
    # an even number of images: the two middle ones are equally far from the median
    gap = np.round(np.abs(z_te[idx] - np.median(z_te[idx])), 9)
    return idx[np.flatnonzero(gap == gap.min())[-1]]  # take the later one, as the reel and the demo do


fig, axes = plt.subplots(2, 3, figsize=(9, 6.2))
for row, j in enumerate([typical(1), typical(0)]):
    c = Xte[j] * w
    axes[row, 0].imshow(Xte[j].reshape(28, 28), cmap="gray_r")
    axes[row, 0].set_title(f"MNIST test #{idx_te[j]}: x", fontsize=10)
    axes[row, 0].axis("off")
    show_w(axes[row, 1], w, "w")
    show_w(axes[row, 2], c, f"x ⊙ w: z = {num(c.sum() + b, 2)}")
    print(
        f"#{idx_te[j]} (a {POS if yte[j] else NEG}): positive {num(c[c > 0].sum(), 2)}, negative {num(c[c < 0].sum(), 2)}, "
        f"b = {num(b, 1)}  →  z = {num(z_te[j], 2)}  →  “{POS if z_te[j] >= 0 else NEG}”"
    )
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

Only the pixels with ink ($x_i > 0$) contribute to $z$; the white background times any $w_i$ is 0.
The 7's strokes lie on the blue regions, so the positive sum wins; the 1 lies on the orange region, so the negative
sum wins.

## 7 · Where does the model go wrong?
The 14 misclassified test images, with their $z$. A $z$ close to 0 means “almost right”.

In [ ]:
wrong = np.nonzero(pred != yte)[0]
fig, axes = plt.subplots(2, 7, figsize=(10, 3.4))
for ax, j in zip(axes.flat, wrong):
    ax.imshow(Xte[j].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"a {POS if yte[j] else NEG}\nz = {num(z_te[j], 2)}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()
print(f"{(yte[wrong] == 1).sum()} 7s misclassified as 1s, {(yte[wrong] == 0).sum()} 1s misclassified as 7s")

10 of the 14 errors are 7s, most of them with a short horizontal bar or an almost vertical stem, so their ink falls
on the orange band of the 1; a few others are written far off to one side. A neuron has only a single “template”
$w$, so handwriting styles far from that template get misclassified.

The distribution of $z$ for the two classes shows the same thing: most images lie far from the threshold 0, and only
a few are on the wrong side.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.linspace(z_te.min(), z_te.max(), 60)
ax.hist(z_te[yte == 0], bins, color=ORANGE, alpha=0.75, label=f"images of {NEG}s")
ax.hist(z_te[yte == 1], bins, color=BLUE, alpha=0.75, label=f"images of {POS}s", hatch="//", edgecolor="white", lw=0)
ax.axvline(0, color="black", lw=1)
ax.set(xlabel="z", ylabel="number of test images", title="z of every test image; threshold z = 0")
ax.legend(frameon=False)
fmt_axes(plt.gcf())
plt.show()

## 8 · Try changing things

The cells below rerun the same `train()` function with other settings. You can change the values and run them again.

### 🤔 Predict first: the learning rate $\eta$
If you change $\eta = 0.1$ to $0.01$ or $1$, how do the number of updates and the accuracy change?

In [ ]:
for eta in [0.01, 0.1, 1.0]:
    w_e, b_e, log_e = train(Xtr, ytr, eta=eta, seed=0)
    acc_e = (step(Xte @ w_e + b_e) == yte).mean()
    print(
        f"η = {num(eta, 2):>4}: {len(log_e['seen'])} updates, {pct(acc_e)} correct, "
        f"w/η identical to η = 0.1: {np.allclose(w_e / eta, w / 0.1)}"
    )

Nothing changes. Because $w$ and $b$ start at 0 and every update adds $\pm\eta x$, $w$ is always $\eta$ times a sum of
images; and the sign of $z$ doesn't change when $z$ is multiplied by a positive number. For a perceptron that starts
from 0, $\eta$ only changes the size of $w$, never a decision. (In Lesson 05, with gradient descent, $\eta$ will
matter.)

### The order of the images (seed)
The order in which the images are visited does matter: each update depends on the current $w$.

In [ ]:
for seed in range(5):
    w_s, b_s, log_s = train(Xtr, ytr, seed=seed)
    print(f"seed {seed}: {len(log_s['seen'])} updates, {pct((step(Xte @ w_s + b_s) == yte).mean())} correct")

From 144 to 163 updates, and from 98.94% to 99.49% correct. When you compare two models, a difference smaller than
this spread between seeds doesn't yet tell you which one is better.

### Training until there are no mistakes left
If the two classes **can be separated by a straight line** (here, a hyperplane in 784-dimensional space),
Novikoff's convergence theorem (1962) guarantees that the perceptron learning rule stops after a finite number of
updates. Let's run epoch after epoch until one epoch misclassifies no training image:

In [ ]:
t0 = time.perf_counter()
w_c, b_c, log_c = train(Xtr, ytr, epochs=200)
errs = log_c["epoch_errors"]
print(
    f"{len(errs)} epochs, {num(len(log_c['seen']), 0)} updates ({num(time.perf_counter() - t0, 1)} s); errors per epoch: {errs[:6]} … {errs[-3:]}"
)
print(
    f"training accuracy {pct((step(Xtr @ w_c + b_c) == ytr).mean())}, test accuracy {pct((step(Xte @ w_c + b_c) == yte).mean())}"
)
assert errs[-1] == 0

The 7/1 set is perfectly separable: after 93 epochs, the model gets 100% of the training images right. But on new
images it only goes from 99.35% to 99.63%, for 93 times the time. Getting the whole training set right doesn't mean
getting every unseen image right (Lesson 08).

### Other pairs of digits
7 and 1 look very different. Change `pair` to try harder pairs, for example 4 and 9, or 3 and 8:

In [ ]:
for pair in [(7, 1), (0, 1), (3, 8), (4, 9), (5, 3)]:
    Xa, ya, _ = binary_set(train_img, train_lab, *pair)
    Xb, yb, _ = binary_set(test_img, test_lab, *pair)
    w_p, b_p, log_p = train(Xa, ya)
    print(
        f"{pair[0]} or {pair[1]}: {num(len(ya), 0)} images, {len(log_p['seen'])} updates, test accuracy {pct((step(Xb @ w_p + b_p) == yb).mean())}"
    )

For pairs that look alike (4/9, 3/8, 5/3), one epoch reaches only 93–95% and needs 5–6 times as many updates. A
single linear template isn't enough to tell apart two digits that share many strokes.

## 9 · The limit: XOR

A neuron's decision boundary $z = 0$ is a straight line (in 2 dimensions). The XOR problem has 4 points:
$(0,0)$ and $(1,1)$ with label 0, $(0,1)$ and $(1,0)$ with label 1. No straight line separates them, so the
perceptron learning rule keeps updating forever without ever getting all 4 right (Minsky & Papert, 1969). We run it
exactly as the reel does: start from $w = (0.3, -0.2)$, $b = 0.1$, and at each step update the first misclassified
point after the one just updated.

In [ ]:
P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y = np.array([0, 1, 1, 0])
w2, b2 = np.array([0.3, -0.2]), 0.1
history, cursor = [], 0
for _ in range(16):
    correct = step(P @ w2 + b2) == Y
    history.append((w2.copy(), b2, int(correct.sum())))
    if correct.all():
        break
    j = next((cursor + q) % 4 for q in range(4) if not correct[(cursor + q) % 4])
    d = Y[j] - int(P[j] @ w2 + b2 >= 0)
    w2, b2 = w2 + 0.1 * d * P[j], b2 + 0.1 * d
    cursor = (j + 1) % 4
history.append((w2.copy(), b2, int((step(P @ w2 + b2) == Y).sum())))
print("points classified correctly after each update:", [h[2] for h in history])
assert max(h[2] for h in history) < 4

Never more than 3/4. The figure below shows the 4 points (orange circles = label 0, blue squares = label 1), a few
of the lines $z = 0$ along the way (gray) and the final line (green; the light blue region is where the neuron
predicts 1). Every line leaves at least one point on the wrong side.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-0.5, 1.5, 201), np.linspace(-0.5, 1.5, 201))
grid = np.c_[g1.ravel(), g2.ravel()]
fig, ax = plt.subplots(figsize=(4.4, 4.4))
for wk, bk, _ in history[:-1:4]:  # a few lines along the way (a line z = 0 can be vertical, so draw it as a contour)
    ax.contour(g1, g2, (grid @ wk + bk).reshape(g1.shape), levels=[0], colors=GREY, linewidths=1)
wl, bl, n_ok = history[-1]
zl = (grid @ wl + bl).reshape(g1.shape)
ax.contourf(g1, g2, zl, levels=[0, np.inf], colors=[BLUE], alpha=0.12)  # the region predicted 1
ax.contour(g1, g2, zl, levels=[0], colors=GREEN, linewidths=2.5)
for p, lab in zip(P, Y):
    ax.scatter(*p, s=160, marker="s" if lab else "o", color=BLUE if lab else ORANGE, edgecolor="white", lw=2, zorder=3)
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x₁", ylabel="x₂")
ax.set_title(f"XOR: the final line (green) gets {n_ok}/4 right", fontsize=11)
ax.set_aspect("equal")
fmt_axes(plt.gcf())
plt.show()

The solution is to add a **hidden layer** with a nonlinear activation function:
[Lesson 03 · Multilayer networks & the XOR problem](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/).

## 10 · From one neuron to a layer: $z = xW + b$

Put $M$ neurons side by side, each with its own column of weights, and you get a $784 \times M$ matrix $W$.
One image (a row vector $x$ of size $1 \times 784$) gives $M$ values at once: $z = xW + b$, at a cost of
$784 \cdot M$ MACs. Stack many images as the rows of $X$, and the whole set is computed with one matrix
multiplication, $Z = XW + b$.

Our neuron is the case $M = 1$. The MLP that MLSysBook (and the course, from Lesson 02 on) uses for MNIST is
784 → 128 → 64 → 10:

In [ ]:
layers = [784, 128, 64, 10]
total = 0
for n_in, n_out in itertools.pairwise(layers):
    total += n_in * n_out
    print(f"layer {n_in:>3} → {n_out:>3}: W of size {n_in}×{n_out}, {num(n_in * n_out, 0):>7} MACs")
print(f"total: {num(total, 0)} MACs per image, against 784 for one neuron")
assert total == 109184

A layer of 10 neurons (one neuron per digit; pick the neuron with the largest $z$) can already read all 10 digits.
The perceptron learning rule extends naturally: on a wrong prediction, add $\eta x$ to the column of the correct digit
and subtract $\eta x$ from the column of the digit it was mistaken for. One epoch over the 60,000 images:

In [ ]:
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)
X_all_te, y_all_te = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
W, b10, fixes = np.zeros((784, 10)), np.zeros(10), 0
for i in np.random.default_rng(0).permutation(len(y_all)):
    k = int(np.argmax(X_all[i] @ W + b10))
    if k != y_all[i]:
        W[:, y_all[i]] += 0.1 * X_all[i]
        b10[y_all[i]] += 0.1
        W[:, k] -= 0.1 * X_all[i]
        b10[k] -= 0.1
        fixes += 1
Z = X_all_te @ W + b10  # 10,000 images × 10 neurons, one matrix multiplication
print(
    f"Z has shape {Z.shape}; {num(fixes, 0)} updates; {pct((Z.argmax(1) == y_all_te).mean(), 1)} correct on the 10,000 test images"
)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.5))
for d, ax in enumerate(axes):
    show_w(ax, W[:, d], f"column {d}")
fmt_axes(plt.gcf())
plt.show()

88.7% on 10 digits: good, but the course's 784 → 128 → 64 → 10 network reaches 97.27% (the Lesson 04 demo). Each
column of $W$ is still just one linear template for one digit (figure above). To do better, the network needs hidden
layers, smooth activation functions and learning by gradient: that is what Chapters 1–2 cover.

## Summary
- A perceptron computes $z = \sum_i x_i w_i + b$ (784 MACs for one MNIST image), then decides with a step function.
- Rosenblatt's learning rule updates only on a mistake: $w \leftarrow w + \eta (y - \hat y) x$. One epoch,
  146 updates: 47.53% → 99.35%.
- The weights can be read as an image: blue is the strokes of a “7”, orange those of a “1”.
- Starting from 0, $\eta$ changes no decision; the order of the images does (98.94–99.49% over 5 seeds).
- On linearly separable data the rule converges (Novikoff); on XOR it never does.
- Many neurons make a layer, $z = xW + b$: a matrix multiplication.

## Review questions
1. Why, with $w = 0$, $b = 0$, does the neuron predict “7” for every image?
2. A pixel has a large positive $w_i$. Which way does ink at that pixel push the decision?
3. After training, 297 weights are still exactly 0. Why?
4. If you multiply $\eta$ by 10, do the predictions change? Why?
5. How many MACs per image does a layer of $M = 10$ neurons cost on 784-pixel images?
6. Why can no perceptron solve XOR?

<details><summary>Hints and answers</summary>

1. $z = 0$ for every image, and $f(0) = 1$ (“7”).
2. Toward “7”: $x_i > 0$ times $w_i > 0$ makes $z$ larger.
3. Each update adds $\pm\eta x$; the pixels with no ink in any of the 146 misclassified images never get anything
   added.
4. No: $w$ and $b$ are both multiplied by 10, so $z$ is multiplied by 10, and the sign of $z$ stays the same.
5. $784 \times 10 = 7{,}840$ MACs.
6. A neuron's decision boundary is a straight line, and no straight line puts $(0,1)$, $(1,0)$ on one side
   and $(0,0)$, $(1,1)$ on the other.
</details>

## Further reading
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html): the MNIST example and the notation of this lesson.
- F. Rosenblatt (1958), [The Perceptron](https://doi.org/10.1037/h0042519).
- A. Novikoff (1962), [On convergence proofs for perceptrons](https://web.engr.oregonstate.edu/~huanlian/teaching/ML/2018spring/extra/novikoff-1963.pdf).
- M. Minsky & S. Papert (1969), [Perceptrons](https://doi.org/10.7551/mitpress/11301.001.0001).
- Y. Freund & R. Schapire (1999), [Large margin classification using the perceptron algorithm](https://doi.org/10.1023/A:1007662407062):
  the voted perceptron, an improvement for data that is not separable.

**Next lesson:** [Lesson 02 · Activation functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-kich-hoat/): why the step function is replaced with smooth functions.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")